# FHOPS Onboarding — Solve & Compare

This notebook solves `tiny7` and `small21` with the simulated annealing (SA)
heuristic via both the Python API (`fhops.optimization.heuristics.solve_sa`)
and the CLI (`fhops solve-heur`), then compares objectives, KPIs, and
schedule summaries.

## What a heuristic solve does

1. **Greedy seed** — an initial feasible assignment of machines to blocks.
2. **Neighbourhood search** — swap, move, block-insertion, cross-exchange,
   mobilisation-shake operators explore alternatives.
3. **Simulated annealing** — probabilistic acceptance of worse solutions
   enables escape from local optima; cooling rate controls the transition.
4. **Full repair + scoring** — each candidate is repaired to feasibility
   before its objective is evaluated against production + mobilisation.

## Budget

- `tiny7`: 50 SA iterations, seed 7.
- `small21`: 100 SA iterations, seed 7.

No licensed solver (Gurobi, CPLEX) is required — the SA heuristic runs with
the bundled HiGHS/Pyomo dependencies.

In [1]:
import sys
import tempfile
from pathlib import Path

working_dir = Path.cwd()
examples_dir = working_dir if (working_dir / "notebook_support.py").exists() else working_dir / "examples"
sys.path.insert(0, str(examples_dir.resolve()))
from notebook_support import (
    find_repo_root,
    discover_scenarios,
    summarise_scenario,
    summarise_schedule,
    validate_schedule,
    run_fhops_cli,
)
from fhops.scenario.io import load_scenario
from fhops.scenario.contract import Problem
from fhops.optimization.heuristics import solve_sa

repo_root = find_repo_root()
print(f'Repo root: {repo_root}')

Repo root: /srv/shared-data/gep/jupyterhub04-projects/fhops


In [2]:
scenarios = discover_scenarios()
for kind, path in scenarios:
    s = summarise_scenario(path)
    print(f'{kind.value:20s}  name={s["name"]}  days={s["num_days"]}  '
          f'blocks={s["num_blocks"]}  machines={s["num_machines"]}')

tiny7                 name=FHOPS Tiny7  days=7  blocks=2  machines=9
small21               name=FHOPS Small21  days=21  blocks=6  machines=9
med42                 name=FHOPS Medium42  days=42  blocks=12  machines=9
synthetic-small       name=synthetic-small  days=112  blocks=4  machines=2
synthetic-medium      name=synthetic-medium  days=112  blocks=8  machines=4
synthetic-large       name=synthetic-large  days=112  blocks=16  machines=6


In [7]:
solve_results = {}
for kind, subpath in [('tiny7', 'tiny7'), ('small21', 'small21')]:
    sc = load_scenario(str(repo_root / 'examples' / subpath / 'scenario.yaml'))
    pb = Problem.from_scenario(sc)
    iters = 50 if kind == 'tiny7' else 100
    print(f'Solving {kind} with {iters} iterations...')
    res = solve_sa(pb, iters=iters, seed=7)
    solve_results[kind] = {
        'objective': res['objective'],
        'assignments': res['assignments'],
        'meta': res['meta'],
    }
    v = summarise_schedule(res['assignments'])
    print(f'  objective={res["objective"]:.3f}  rows={v["rows"]}  '
          f'blocks={v["blocks_completed"]}  machines={v["machines_used"]}')

Solving tiny7 with 50 iterations...
  objective=4306.523  rows=23  blocks=2  machines=8
Solving small21 with 100 iterations...
  objective=15574.425  rows=90  blocks=6  machines=9


## Solve via the CLI

The CLI `solve-heur` writes the assignments to a CSV and prints KPIs.

In [4]:
import pandas as pd

cli_results = {}
with tempfile.TemporaryDirectory() as tmpdir:
    for kind, subpath in [('tiny7', 'tiny7'), ('small21', 'small21')]:
        out_csv = Path(tmpdir) / f'{kind}_sa.csv'
        iters = 50 if kind == 'tiny7' else 100
        print(f'CLI solve-heur {kind} ({iters} iters)...')
        result = run_fhops_cli(
            'solve-heur', str(repo_root / 'examples' / subpath / 'scenario.yaml'),
            '--out', str(out_csv),
            '--iters', str(iters),
            '--seed', '7',
        )
        df = pd.read_csv(out_csv)
        v = summarise_schedule(df)
        cli_results[kind] = {'assignments': df, 'stdout': result['stdout']}
        print(f'  rows={v["rows"]}  blocks={v["blocks_completed"]}  machines={v["machines_used"]}')

CLI solve-heur tiny7 (50 iters)...
  rows=23  blocks=2  machines=8
CLI solve-heur small21 (100 iters)...
  rows=90  blocks=6  machines=9


## Compare objectives and KPIs

We compute KPIs for the API-solved schedules and display compact summaries.

In [5]:
from fhops.evaluation import compute_kpis

comparison = []
for kind, subpath in [('tiny7', 'tiny7'), ('small21', 'small21')]:
    sc = load_scenario(str(repo_root / 'examples' / subpath / 'scenario.yaml'))
    pb = Problem.from_scenario(sc)
    api_res = solve_results[kind]
    cli_res = cli_results[kind]
    
    api_kpis = compute_kpis(pb, api_res['assignments'])
    cli_kpis = compute_kpis(pb, cli_res['assignments'])
    
    comparison.append({
        'scenario': kind,
        'api_objective': api_res['objective'],
        'cli_total_production': cli_kpis['total_production'],
        'cli_completed_blocks': cli_kpis['completed_blocks'],
        'cli_mobilisation_cost': cli_kpis['mobilisation_cost'],
        'cli_makespan_day': cli_kpis['makespan_day'],
        'api_completed_blocks': api_kpis['completed_blocks'],
        'api_mobilisation_cost': api_kpis['mobilisation_cost'],
    })

pd.DataFrame(comparison).round(3)

,scenario,api_objective,cli_total_production,cli_completed_blocks,cli_mobilisation_cost,cli_makespan_day,api_completed_blocks,api_mobilisation_cost
0,tiny7,4306.523,4414.703,2.0,585.64,7,2.0,585.64
1,small21,15574.425,15966.967,6.0,2564.40,17,6.0,2564.40


## Schedule previews

A compact look at the API-solved assignments for tiny7.

In [6]:
tiny7_assign = solve_results['tiny7']['assignments']
tiny7_assign[['machine_id', 'block_id', 'day', 'shift_id']].head(20)

,machine_id,block_id,day,shift_id
0,H1,B02,1,S1
2,H2,B02,1,S1
1,H1,B01,2,S1
3,H2,B01,2,S1
4,H3,B02,2,S1
5,H3,B01,3,S1
8,H4,B02,3,S1
12,H5,B02,3,S1
6,H3,B01,4,S1
9,H4,B01,4,S1


- Next: `03_fhops_playback_kpis.ipynb` runs playback on these schedules and
